# 01 — Data Exploration

## Week: 3

### Project: CityFix – Civic Service Analytics

### Objective
The objective of this notebook is to explore the raw CityFix datasets before implementing the Bronze ingestion layer.

During this exploration we will:

- Load the raw source files from the Databricks Volume
- Inspect schemas and data types
- Display sample records
- Measure physical row counts
- Check business key uniqueness
- Identify missing values
- Validate relationships between datasets
- Explore important business attributes using Spark SQL
- Create one demonstration Bronze table for Week 3 only

**Week 3 Scope:** Data exploration and understanding only.

**Week 4 Scope:** Bronze ingestion pipeline implementation.

In [0]:
from pyspark.sql.functions import *

In [0]:
requests_df = spark.read.option("header", "true").csv("<volume_path>/requests_sample.csv")

agencies_df = spark.read.option("header", "true").csv("<volume_path>/agencies_sample.csv")

boroughs_df = spark.read.option("header", "true").csv("<volume_path>/boroughs_sample.csv")

categories_df = spark.read.option("header", "true").csv("<volume_path>/categories_sample.csv")

zip_df = spark.read.option("header", "true").csv("<volume_path>/zip_geography_sample.csv")

In [0]:
agencies_df = spark.read.option("header","true").csv("/Volumes/cityfix/cityfix/cityfix-zenaiz/raw/agencies.csv")

boroughs_df = spark.read.option("header","true").csv("/Volumes/cityfix/cityfix/cityfix-zenaiz/raw/boroughs.csv")

categories_df = spark.read.option("header","true").csv("/Volumes/cityfix/cityfix/cityfix-zenaiz/raw/categories.csv")

requests_df = spark.read.option("header","true").csv("/Volumes/cityfix/cityfix/cityfix-zenaiz/raw/requests.csv")

zip_df = spark.read.option("header","true").csv("/Volumes/cityfix/cityfix/cityfix-zenaiz/raw/zip_geography.csv")

In [0]:
display(agencies_df)

agency_code,agency_name,service_scope,active_flag,effective_from
PWD,Public Works Department,"Roads, drains, street infrastructure",True,2024-01-01
SAN,Sanitation Services,Waste collection and public cleanliness,True,2024-01-01
WAT,Water Services,"Water supply, leakage and sewer support",True,2024-01-01
TRN,Transport Operations,Traffic assets and public transport stops,True,2024-01-01
PKS,Parks and Urban Green,"Parks, trees and public spaces",True,2024-01-01
HSG,Housing Services,Municipal housing and building maintenance,True,2024-01-01
HLT,Public Health Services,Public health and sanitation inspections,True,2024-01-01
ENV,Environmental Services,"Noise, air, water and nuisance monitoring",True,2024-01-01
LIC,Licensing and Permits,Municipal permits and licence queries,True,2024-01-01
FIR,Fire and Safety Services,Fire-safety and emergency infrastructure,True,2024-01-01


In [0]:
display(categories_df)

category_code,complaint_category,complaint_type,default_descriptor,default_agency_code,standard_sla_hours,active_flag,priority_hint
CAT-ROAD-01,Roads and Footpaths,Pothole,Road surface damage,PWD,72,True,Routine
CAT-ROAD-02,Roads and Footpaths,Footpath Damage,Broken or obstructed footpath,PWD,120,True,Routine
CAT-ROAD-03,Roads and Footpaths,Streetlight Fault,Lamp not working or flickering,PWD,48,True,Medium
CAT-WASTE-01,Waste and Cleanliness,Missed Waste Collection,Scheduled collection missed,SAN,24,True,Medium
CAT-WASTE-02,Waste and Cleanliness,Illegal Dumping,Waste dumped in public area,SAN,48,True,Medium
CAT-WASTE-03,Waste and Cleanliness,Overflowing Bin,Public bin requires service,SAN,12,True,High
CAT-WATER-01,Water and Sewer,Water Leakage,Visible supply-line leakage,WAT,12,True,High
CAT-WATER-02,Water and Sewer,Low Water Pressure,Reduced supply pressure,WAT,24,True,Medium
CAT-WATER-03,Water and Sewer,Sewer Blockage,Drain or sewer obstruction,WAT,8,True,High
CAT-TRAFFIC-01,Traffic and Transport,Traffic Signal Fault,Signal not operating normally,TRN,8,True,High


In [0]:
display(requests_df.limit(20))

physical_record_key,unique_key,created_date,due_date,closed_date,agency_code,agency_name_raw,complaint_category,complaint_type,descriptor,borough_code,borough,zip_code,latitude,longitude,location_type,channel,status,priority,resolution_description,sla_hours,source_system,source_file,batch_id,ingestion_timestamp,record_hash
REC-000000001,CFX-2025-000000001,2025-05-07T17:19:07Z,2025-05-10T17:19:07Z,2025-05-11T14:02:55Z,HLT,Public Health Services,Public Health,Mosquito Breeding,Potential breeding location,WST,West Borough,510082,17.421957,78.342517,Street,Web Portal,Closed,Normal,Asset repaired or restored,72,CITYFIX_PORTAL,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,5b4c14aa552dcbf9
REC-000000002,CFX-2025-000000002,2025-11-14T15:59:38Z,2025-11-16T05:59:38Z,2025-11-15T19:57:50Z,ENV,Environmental Services,Environment and Nuisance,Noise Nuisance,Persistent public noise concern,WST,West Borough,510083,17.427382,78.365933,Municipal Building,Mobile App,Closed,High,Resolved through agency coordination,38,CITYFIX_PORTAL,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,b537fb27a6db4c80
REC-000000003,CFX-2025-000000003,2025-08-11T17:54:26Z,2025-08-13T17:54:26Z,2025-08-13T14:40:02Z,PWD,Public Works Department,Roads and Footpaths,Streetlight Fault,Lamp not working or flickering,STH,South Borough,510065,17.331992,78.439868,Municipal Building,Web Portal,Closed,Normal,Service completed,48,CITYFIX_MOBILE,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,78401aff2f7885c2
REC-000000004,CFX-2025-000000004,2025-07-19T00:04:05Z,2025-07-19T10:04:05Z,2025-07-21T09:46:41Z,PKS,Parks and Urban Green,Parks and Public Space,Fallen Tree or Branch,Tree or branch obstructing area,CTR,Central Borough,510045,17.451505,78.452124,Public Space,Email,Closed,High,Issue inspected and resolved,10,CITYFIX_MOBILE,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,0f8f7e0c9d82791e
REC-000000005,CFX-2025-000000005,2025-05-18T12:24:53Z,2025-05-25T06:24:53Z,2025-06-01T13:48:53Z,PWD,Public Works Department,Roads and Footpaths,Footpath Damage,Broken or obstructed footpath,EST,East Borough,510027,17.460283,78.63478,Commercial Area,Call Centre,Closed,Low,Service completed,162,CITYFIX_CONTACT_CENTRE,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,c328440dd5090cae
REC-000000006,CFX-2025-000000006,2025-03-07T16:54:59Z,2025-03-08T02:54:59Z,2025-03-07T23:44:47Z,SAN,Sanitation Services,Waste and Cleanliness,Overflowing Bin,Public bin requires service,CTR,Central Borough,510047,17.461696,78.507866,Public Space,Call Centre,Closed,High,Asset repaired or restored,10,CITYFIX_PORTAL,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,d07f2a98f7927b2e
REC-000000007,CFX-2025-000000007,2025-03-13T11:25:22Z,2025-03-14T11:25:22Z,2025-03-14T10:00:09Z,SAN,Sanitation Services,Waste and Cleanliness,Missed Waste Collection,Scheduled collection missed,STH,South Borough,510065,17.337046,78.438212,Residential Area,Call Centre,Closed,Normal,Service completed,24,CITYFIX_PORTAL,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,0b19693eff9e6e86
REC-000000008,CFX-2025-000000008,2025-07-14T15:22:08Z,2025-07-17T15:22:08Z,2025-07-16T01:25:44Z,TRN,Transport Operations,Traffic and Transport,Damaged Bus Stop,Shelter or stop asset damaged,WST,West Borough,510081,17.405146,78.315197,Street,Call Centre,Closed,Normal,Resolved through agency coordination,72,CITYFIX_CONTACT_CENTRE,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,a832760d23f52bed
REC-000000009,CFX-2025-000000009,2025-12-30T13:08:37Z,2025-12-30T16:08:37Z,2025-12-30T23:27:13Z,FIR,Fire and Safety Services,Fire and Public Safety,Blocked Fire Access,Fire access obstructed,NTH,North Borough,510005,17.538395,78.446193,Commercial Area,Mobile App,Closed,High,Issue inspected and resolved,3,CITYFIX_CONTACT_CENTRE,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,1d7c28e99c470a7f
REC-000000010,CFX-2025-000000010,2025-08-15T09:03:08Z,2025-08-22T09:03:08Z,2025-08-18T18:30:44Z,PKS,Parks and Urban Green,Parks and Publ

In [0]:
display(zip_df)

zip_code,borough_code,borough_name,service_zone_name,centroid_latitude,centroid_longitude,min_latitude,max_latitude,min_longitude,max_longitude
510001,NTH,North Borough,NTH Zone 1,17.4935,78.4385,17.4755,17.5115,78.4185,78.4585
510002,NTH,North Borough,NTH Zone 2,17.4995,78.4635,17.4815,17.5175,78.4435,78.4835
510003,NTH,North Borough,NTH Zone 3,17.5055,78.4885,17.4875,17.5235,78.4685,78.5085
510004,NTH,North Borough,NTH Zone 4,17.5115,78.5135,17.4935,17.5295,78.4935,78.5335
510005,NTH,North Borough,NTH Zone 5,17.5285,78.4465,17.5105,17.5465,78.4265,78.4665
510006,NTH,North Borough,NTH Zone 6,17.5345,78.4715,17.5165,17.5525,78.4515,78.4915
510007,NTH,North Borough,NTH Zone 7,17.5405,78.4965,17.5225,17.5585,78.4765,78.5165
510008,NTH,North Borough,NTH Zone 8,17.5465,78.5215,17.5285,17.5645,78.5015,78.5415
510021,EST,East Borough,EST Zone 1,17.4035,78.5685,17.3855,17.4215,78.5485,78.5885
510022,EST,East Borough,EST Zone 2,17.4095,78.5935,17.3915,17.4275,78.5735,78.6135


In [0]:
agencies_df = spark.read.option("header", "true").csv(
    "/Volumes/cityfix/cityfix/cityfix-zenaiz/raw/agencies.csv"
)

In [0]:
display(agencies_df)

agency_code,agency_name,service_scope,active_flag,effective_from
PWD,Public Works Department,"Roads, drains, street infrastructure",True,2024-01-01
SAN,Sanitation Services,Waste collection and public cleanliness,True,2024-01-01
WAT,Water Services,"Water supply, leakage and sewer support",True,2024-01-01
TRN,Transport Operations,Traffic assets and public transport stops,True,2024-01-01
PKS,Parks and Urban Green,"Parks, trees and public spaces",True,2024-01-01
HSG,Housing Services,Municipal housing and building maintenance,True,2024-01-01
HLT,Public Health Services,Public health and sanitation inspections,True,2024-01-01
ENV,Environmental Services,"Noise, air, water and nuisance monitoring",True,2024-01-01
LIC,Licensing and Permits,Municipal permits and licence queries,True,2024-01-01
FIR,Fire and Safety Services,Fire-safety and emergency infrastructure,True,2024-01-01


In [0]:
boroughs_df = spark.read.option("header","true").csv("/Volumes/cityfix/cityfix/cityfix-zenaiz/raw/boroughs.csv")

categories_df = spark.read.option("header","true").csv("/Volumes/cityfix/cityfix/cityfix-zenaiz/raw/categories.csv")

requests_df = spark.read.option("header","true").csv("/Volumes/cityfix/cityfix/cityfix-zenaiz/raw/requests.csv")

zip_df = spark.read.option("header","true").csv("/Volumes/cityfix/cityfix/cityfix-zenaiz/raw/zip_geography.csv")

In [0]:
display(boroughs_df)

borough_code,borough_name,borough_short_name,reporting_order,active_flag
NTH,North Borough,NB,1,True
EST,East Borough,EB,2,True
CTR,Central Borough,CB,3,True
STH,South Borough,SB,4,True
WST,West Borough,WB,5,True
UNK,Unknown / Unassigned,UN,99,True


In [0]:
display(categories_df)

category_code,complaint_category,complaint_type,default_descriptor,default_agency_code,standard_sla_hours,active_flag,priority_hint
CAT-ROAD-01,Roads and Footpaths,Pothole,Road surface damage,PWD,72,True,Routine
CAT-ROAD-02,Roads and Footpaths,Footpath Damage,Broken or obstructed footpath,PWD,120,True,Routine
CAT-ROAD-03,Roads and Footpaths,Streetlight Fault,Lamp not working or flickering,PWD,48,True,Medium
CAT-WASTE-01,Waste and Cleanliness,Missed Waste Collection,Scheduled collection missed,SAN,24,True,Medium
CAT-WASTE-02,Waste and Cleanliness,Illegal Dumping,Waste dumped in public area,SAN,48,True,Medium
CAT-WASTE-03,Waste and Cleanliness,Overflowing Bin,Public bin requires service,SAN,12,True,High
CAT-WATER-01,Water and Sewer,Water Leakage,Visible supply-line leakage,WAT,12,True,High
CAT-WATER-02,Water and Sewer,Low Water Pressure,Reduced supply pressure,WAT,24,True,Medium
CAT-WATER-03,Water and Sewer,Sewer Blockage,Drain or sewer obstruction,WAT,8,True,High
CAT-TRAFFIC-01,Traffic and Transport,Traffic Signal Fault,Signal not operating normally,TRN,8,True,High


In [0]:
display(requests_df.limit(20))

physical_record_key,unique_key,created_date,due_date,closed_date,agency_code,agency_name_raw,complaint_category,complaint_type,descriptor,borough_code,borough,zip_code,latitude,longitude,location_type,channel,status,priority,resolution_description,sla_hours,source_system,source_file,batch_id,ingestion_timestamp,record_hash
REC-000000001,CFX-2025-000000001,2025-05-07T17:19:07Z,2025-05-10T17:19:07Z,2025-05-11T14:02:55Z,HLT,Public Health Services,Public Health,Mosquito Breeding,Potential breeding location,WST,West Borough,510082,17.421957,78.342517,Street,Web Portal,Closed,Normal,Asset repaired or restored,72,CITYFIX_PORTAL,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,5b4c14aa552dcbf9
REC-000000002,CFX-2025-000000002,2025-11-14T15:59:38Z,2025-11-16T05:59:38Z,2025-11-15T19:57:50Z,ENV,Environmental Services,Environment and Nuisance,Noise Nuisance,Persistent public noise concern,WST,West Borough,510083,17.427382,78.365933,Municipal Building,Mobile App,Closed,High,Resolved through agency coordination,38,CITYFIX_PORTAL,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,b537fb27a6db4c80
REC-000000003,CFX-2025-000000003,2025-08-11T17:54:26Z,2025-08-13T17:54:26Z,2025-08-13T14:40:02Z,PWD,Public Works Department,Roads and Footpaths,Streetlight Fault,Lamp not working or flickering,STH,South Borough,510065,17.331992,78.439868,Municipal Building,Web Portal,Closed,Normal,Service completed,48,CITYFIX_MOBILE,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,78401aff2f7885c2
REC-000000004,CFX-2025-000000004,2025-07-19T00:04:05Z,2025-07-19T10:04:05Z,2025-07-21T09:46:41Z,PKS,Parks and Urban Green,Parks and Public Space,Fallen Tree or Branch,Tree or branch obstructing area,CTR,Central Borough,510045,17.451505,78.452124,Public Space,Email,Closed,High,Issue inspected and resolved,10,CITYFIX_MOBILE,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,0f8f7e0c9d82791e
REC-000000005,CFX-2025-000000005,2025-05-18T12:24:53Z,2025-05-25T06:24:53Z,2025-06-01T13:48:53Z,PWD,Public Works Department,Roads and Footpaths,Footpath Damage,Broken or obstructed footpath,EST,East Borough,510027,17.460283,78.63478,Commercial Area,Call Centre,Closed,Low,Service completed,162,CITYFIX_CONTACT_CENTRE,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,c328440dd5090cae
REC-000000006,CFX-2025-000000006,2025-03-07T16:54:59Z,2025-03-08T02:54:59Z,2025-03-07T23:44:47Z,SAN,Sanitation Services,Waste and Cleanliness,Overflowing Bin,Public bin requires service,CTR,Central Borough,510047,17.461696,78.507866,Public Space,Call Centre,Closed,High,Asset repaired or restored,10,CITYFIX_PORTAL,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,d07f2a98f7927b2e
REC-000000007,CFX-2025-000000007,2025-03-13T11:25:22Z,2025-03-14T11:25:22Z,2025-03-14T10:00:09Z,SAN,Sanitation Services,Waste and Cleanliness,Missed Waste Collection,Scheduled collection missed,STH,South Borough,510065,17.337046,78.438212,Residential Area,Call Centre,Closed,Normal,Service completed,24,CITYFIX_PORTAL,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,0b19693eff9e6e86
REC-000000008,CFX-2025-000000008,2025-07-14T15:22:08Z,2025-07-17T15:22:08Z,2025-07-16T01:25:44Z,TRN,Transport Operations,Traffic and Transport,Damaged Bus Stop,Shelter or stop asset damaged,WST,West Borough,510081,17.405146,78.315197,Street,Call Centre,Closed,Normal,Resolved through agency coordination,72,CITYFIX_CONTACT_CENTRE,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,a832760d23f52bed
REC-000000009,CFX-2025-000000009,2025-12-30T13:08:37Z,2025-12-30T16:08:37Z,2025-12-30T23:27:13Z,FIR,Fire and Safety Services,Fire and Public Safety,Blocked Fire Access,Fire access obstructed,NTH,North Borough,510005,17.538395,78.446193,Commercial Area,Mobile App,Closed,High,Issue inspected and resolved,3,CITYFIX_CONTACT_CENTRE,requests.csv,CITYFIX-BATCH-2025-A,2026-07-18T00:30:00Z,1d7c28e99c470a7f
REC-000000010,CFX-2025-000000010,2025-08-15T09:03:08Z,2025-08-22T09:03:08Z,2025-08-18T18:30:44Z,PKS,Parks and Urban Green,Parks and Publ

In [0]:
display(zip_df)

zip_code,borough_code,borough_name,service_zone_name,centroid_latitude,centroid_longitude,min_latitude,max_latitude,min_longitude,max_longitude
510001,NTH,North Borough,NTH Zone 1,17.4935,78.4385,17.4755,17.5115,78.4185,78.4585
510002,NTH,North Borough,NTH Zone 2,17.4995,78.4635,17.4815,17.5175,78.4435,78.4835
510003,NTH,North Borough,NTH Zone 3,17.5055,78.4885,17.4875,17.5235,78.4685,78.5085
510004,NTH,North Borough,NTH Zone 4,17.5115,78.5135,17.4935,17.5295,78.4935,78.5335
510005,NTH,North Borough,NTH Zone 5,17.5285,78.4465,17.5105,17.5465,78.4265,78.4665
510006,NTH,North Borough,NTH Zone 6,17.5345,78.4715,17.5165,17.5525,78.4515,78.4915
510007,NTH,North Borough,NTH Zone 7,17.5405,78.4965,17.5225,17.5585,78.4765,78.5165
510008,NTH,North Borough,NTH Zone 8,17.5465,78.5215,17.5285,17.5645,78.5015,78.5415
510021,EST,East Borough,EST Zone 1,17.4035,78.5685,17.3855,17.4215,78.5485,78.5885
510022,EST,East Borough,EST Zone 2,17.4095,78.5935,17.3915,17.4275,78.5735,78.6135


In [0]:
agencies_df.printSchema()

boroughs_df.printSchema()

categories_df.printSchema()

requests_df.printSchema()

zip_df.printSchema()

root
 |-- agency_code: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- service_scope: string (nullable = true)
 |-- active_flag: string (nullable = true)
 |-- effective_from: string (nullable = true)

root
 |-- borough_code: string (nullable = true)
 |-- borough_name: string (nullable = true)
 |-- borough_short_name: string (nullable = true)
 |-- reporting_order: string (nullable = true)
 |-- active_flag: string (nullable = true)

root
 |-- category_code: string (nullable = true)
 |-- complaint_category: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- default_descriptor: string (nullable = true)
 |-- default_agency_code: string (nullable = true)
 |-- standard_sla_hours: string (nullable = true)
 |-- active_flag: string (nullable = true)
 |-- priority_hint: string (nullable = true)

root
 |-- physical_record_key: string (nullable = true)
 |-- unique_key: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- due_dat

In [0]:
print("Agencies:", agencies_df.count())

print("Boroughs:", boroughs_df.count())

print("Categories:", categories_df.count())

print("Requests:", requests_df.count())

print("ZIP Geography:", zip_df.count())

Agencies: 12
Boroughs: 6
Categories: 32
Requests: 180012
ZIP Geography: 40


In [0]:
from pyspark.sql.functions import col, count, when

requests_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in requests_df.columns
]).show()

+-------------------+----------+------------+--------+-----------+-----------+---------------+------------------+--------------+----------+------------+-------+--------+--------+---------+-------------+-------+------+--------+----------------------+---------+-------------+-----------+--------+-------------------+-----------+
|physical_record_key|unique_key|created_date|due_date|closed_date|agency_code|agency_name_raw|complaint_category|complaint_type|descriptor|borough_code|borough|zip_code|latitude|longitude|location_type|channel|status|priority|resolution_description|sla_hours|source_system|source_file|batch_id|ingestion_timestamp|record_hash|
+-------------------+----------+------------+--------+-----------+-----------+---------------+------------------+--------------+----------+------------+-------+--------+--------+---------+-------------+-------+------+--------+----------------------+---------+-------------+-----------+--------+-------------------+-----------+
|                  

In [0]:
requests_df.describe().show()

+-------+-------------------+------------------+--------------------+--------------------+--------------------+-----------+--------------------+------------------+--------------------+--------------------+------------+---------------+-----------------+------------------+-------------------+---------------+-----------+--------+--------+----------------------+------------------+--------------------+------------+--------------------+--------------------+----------------+
|summary|physical_record_key|        unique_key|        created_date|            due_date|         closed_date|agency_code|     agency_name_raw|complaint_category|      complaint_type|          descriptor|borough_code|        borough|         zip_code|          latitude|          longitude|  location_type|    channel|  status|priority|resolution_description|         sla_hours|       source_system| source_file|            batch_id| ingestion_timestamp|     record_hash|
+-------+-------------------+------------------+------